In [ ]:
# (Colab) run in a cell with `!` or in terminal without it
!pip install --upgrade pip
!pip install torch torchaudio transformers datasets soundfile librosa soundfile numpy scikit-learn joblib tqdm git+https://github.com/openai/whisper.git


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-9acji12e
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-9acji12e
  Resolved https://github.com/openai/whisper.git to commit c0d2f624c09dc18e709e37c2ad90c039a4eb72a2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=3d5cbe7a967a83656b0e8059dca6cf5c20eb65927143daf5fb4c1d880c8b6ceb
  Stored in directory: /tmp/pip-ephem-wheel-cache-8509eut0/wheels/c3/03/25/5e0ba78bc27a3a089f137c9f1d92fdfce16d06996c071a016c
Successfully built openai-whisper


In [ ]:
"""
train_tone_finetune.py
- Fine-tunes a wav2vec2 classification head using 5 archetype mp3s + augmentation.
- Saves fine-tuned model to OUTPUT_DIR/tone_wav2vec_finetuned
Usage:
  - Update TRAIN_AUDIO_DIR if needed (path to the folder containing the 5 archetypes).
  - Run: python train_tone_finetune.py
Notes:
  - This freezes the wav2vec2 base and trains only the classifier head (safer for 5 files).
  - GPU strongly recommended.
"""

import os, random, json
from pathlib import Path
import numpy as np
import librosa
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification
from sklearn.preprocessing import LabelEncoder
import joblib

# ---------- CONFIG ----------
TRAIN_AUDIO_DIR = "/content/drive/MyDrive/Hackthon/INNOV8 3.0/INNOV8 3.0"
OUTPUT_DIR = "/content/drive/MyDrive/Hackthon/Output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# mapping filenames -> tone labels (edit if your filenames differ)
tone_labels = {
    "selfcorrect_contradictions.mp3": "confident",
    "rehearsed_evasion.mp3": "static_interference",
    "hedge_and_dodge.mp3": "shouting",
    "drifted_anecdote.mp3": "whispered",
    "buried_confession.mp3": "emotional"
}

SR = 16000
RANDOM_SEED = 42
AUG_PER_FILE = 40         # reduce to 10-20 for faster run
BATCH_SIZE = 8
EPOCHS = 6
LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# ---------- helpers ----------
def load_audio(path, sr=SR):
    y, _ = librosa.load(path, sr=sr, mono=True)
    return y

# augmentation uses keyword args so works with librosa>=0.10 and older
def augment_audio(y):
    # small random chain of augmentations
    if np.random.rand() < 0.5:
        # pitch_shift (keyword args)
        y = librosa.effects.pitch_shift(y=y, sr=SR, n_steps=np.random.uniform(-2, 2))
    if np.random.rand() < 0.5:
        # time_stretch (keyword args)
        rate = np.random.uniform(0.9, 1.1)
        try:
            y = librosa.effects.time_stretch(y=y, rate=rate)
        except Exception:
            # time_stretch can fail if signal too short, fallback
            pass
    if np.random.rand() < 0.6:
        noise = np.random.normal(0, 0.003, size=y.shape)
        y = y + noise
    # amplitude jitter
    if np.random.rand() < 0.6:
        y = y * np.random.uniform(0.8, 1.2)
    return y

# Prepare training audio list + augmented copies
print("Collecting and augmenting audio...")
audio_list = []
labels = []
for fname, label in tone_labels.items():
    path = os.path.join(TRAIN_AUDIO_DIR, fname)
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Cannot find training audio: {path}")
    y = load_audio(path)
    # original
    audio_list.append(y)
    labels.append(label)
    # augmentations
    for _ in range(AUG_PER_FILE):
        y_aug = augment_audio(y)
        audio_list.append(y_aug)
        labels.append(label)

print("Total training examples:", len(audio_list))

# ---------- Processor + Model ----------
print("Loading processor and model...")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
label_encoder = LabelEncoder()
y_enc = label_encoder.fit_transform(labels)
num_labels = len(label_encoder.classes_)
print("Labels:", label_encoder.classes_)

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=num_labels,
)
# freeze base wav2vec2 parameters (train only classification head)
for p in model.wav2vec2.parameters():
    p.requires_grad = False

model.to(DEVICE)

# ---------- Precompute input_values using processor ----------
print("Precomputing input_values (processor) ...")
input_values_list = []
lengths = []
for y in audio_list:
    proc = processor(y, sampling_rate=SR, return_tensors="np", padding=False)
    iv = proc["input_values"][0]        # 1D numpy array
    input_values_list.append(iv.astype(np.float32))
    lengths.append(len(iv))

print("Average frames:", np.mean(lengths), "max frames:", np.max(lengths))

# ---------- Dataset + Collator ----------
class AudioDataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = labels
    def __len__(self):
        return len(self.inputs)
    def __getitem__(self, idx):
        return {"input_values": self.inputs[idx], "label": int(self.labels[idx])}

def collate_batch(batch):
    # batch: list of dicts
    xs = [b["input_values"] for b in batch]
    labs = [b["label"] for b in batch]
    max_len = max([x.shape[0] for x in xs])
    padded = np.zeros((len(xs), max_len), dtype=np.float32)
    attention_masks = np.zeros((len(xs), max_len), dtype=np.int64)
    for i,x in enumerate(xs):
        l = x.shape[0]
        padded[i, :l] = x
        attention_masks[i, :l] = 1
    return {
        "input_values": torch.from_numpy(padded),
        "attention_mask": torch.from_numpy(attention_masks),
        "labels": torch.tensor(labs, dtype=torch.long)
    }

# create dataset and dataloader
dataset = AudioDataset(input_values_list, y_enc)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch, num_workers=2)

# ---------- Training loop ----------
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
model.train()
print("Training on device:", DEVICE)
for epoch in range(EPOCHS):
    total_loss = 0.0
    for batch in loader:
        input_vals = batch["input_values"].to(DEVICE)
        att_mask = batch["attention_mask"].to(DEVICE)
        labels_batch = batch["labels"].to(DEVICE)
        outputs = model(input_values=input_vals, attention_mask=att_mask, labels=labels_batch)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}/{EPOCHS} — loss {avg_loss:.4f}")

# ---------- Save model and artifacts ----------
save_dir = os.path.join(OUTPUT_DIR, "tone_wav2vec_finetuned")
os.makedirs(save_dir, exist_ok=True)
model.save_pretrained(save_dir)
processor.save_pretrained(save_dir)
joblib.dump(label_encoder, os.path.join(save_dir, "label_encoder.pkl"))
print("Saved fine-tuned model to", save_dir)


Total training examples: 205
Loading processor and model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Precomputing input_values (processor) ...
Average frames: 545804.1219512195 max frames: 636501
Training on device: cuda
Epoch 1/6 — loss 1.5978
Epoch 2/6 — loss 1.5440
Epoch 3/6 — loss 1.4913
Epoch 4/6 — loss 1.4340
Epoch 5/6 — loss 1.3736
Epoch 6/6 — loss 1.3011
Saved fine-tuned model to /content/drive/MyDrive/Hackthon/Output/tone_wav2vec_finetuned


In [ ]:
"""
transcribe_and_label.py
- For each .mp3 in the evaluation folder, predict tone and transcribe.
- Saves:
   - /Output/<subject>_sessions.json   (contains per-session transcript+tone+claims)
   - /Output/<subject>_summary.txt     (human readable summary like your example)
Usage:
  - Make sure the fine-tuned model exists at OUTPUT_DIR/tone_wav2vec_finetuned
  - Run: python transcribe_and_label.py
"""

import os, glob, json, re
from pathlib import Path
import librosa
import numpy as np
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification
import joblib
import whisper
from collections import defaultdict

# ---------- CONFIG ----------
EVAL_AUDIO_DIR = "/content/drive/MyDrive/Hackthon/INNOV8 3.0/Evaluation set/audio"
OUTPUT_DIR = "/content/drive/MyDrive/Hackthon/Output"
MODEL_DIR = os.path.join(OUTPUT_DIR, "tone_wav2vec_finetuned")
os.makedirs(OUTPUT_DIR, exist_ok=True)

SR = 16000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---------- Load fine-tuned model + processor + label encoder ----------
processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
model = Wav2Vec2ForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
label_encoder = joblib.load(os.path.join(MODEL_DIR, "label_encoder.pkl"))

# whisper for ASR
whisper_model = whisper.load_model("small")  # change to tiny/medium as you need

# helpers
def load_audio(path, sr=SR):
    y, _ = librosa.load(path, sr=sr, mono=True)
    return y

def predict_tone_from_audio(y):
    proc = processor(y, sampling_rate=SR, return_tensors="pt", padding=True)
    input_vals = proc["input_values"].to(DEVICE)
    att_mask = proc["attention_mask"].to(DEVICE) if "attention_mask" in proc else None
    with torch.no_grad():
        out = model(input_values=input_vals, attention_mask=att_mask)
        logits = out.logits.cpu().numpy()[0]
    pred = int(np.argmax(logits))
    label = label_encoder.inverse_transform([pred])[0]
    return label, float(np.max(torch.softmax(torch.tensor(logits), dim=-1).numpy()))

# group files by subject
files = sorted(glob.glob(os.path.join(EVAL_AUDIO_DIR, "*.mp3")))
subjects = defaultdict(list)
for f in files:
    stem = Path(f).stem
    parts = stem.split("_")
    if parts[-1].isdigit():
        subj = "_".join(parts[:-1])
    else:
        subj = "_".join(parts[:2]) if len(parts)>=2 else parts[0]
    subjects[subj].append(f)

# process each subject
for subj, flist in subjects.items():
    flist = sorted(flist)
    sessions = []
    for i,f in enumerate(flist, start=1):
        print(f"[{subj}] Processing {Path(f).name} ...")
        y = load_audio(f, SR)
        tone_label, tone_conf = predict_tone_from_audio(y)
        # transcript: try to find existing txt file (same basename) else run whisper
        txt_file = os.path.splitext(f)[0] + ".txt"
        if os.path.isfile(txt_file):
            transcript = open(txt_file, "r", encoding="utf-8").read().strip()
        else:
            # whisper transcription
            res = whisper_model.transcribe(f)
            transcript = res.get("text","").strip()
        sessions.append({
            "session_id": f"{subj}_{i}",
            "file": Path(f).name,
            "tone": tone_label,
            "tone_confidence": tone_conf,
            "transcript": transcript
        })

    # save sessions JSON
    sessions_path = os.path.join(OUTPUT_DIR, f"{subj}_sessions.json")
    with open(sessions_path, "w", encoding="utf-8") as fh:
        json.dump({"shadow_id": subj, "sessions": sessions}, fh, indent=2)
    print("Saved sessions JSON:", sessions_path)

    # build human-readable summary text (first ~220 chars per session)
    summary_lines = []
    summary_lines.append(f"Subject: {subj}")
    summary_lines.append("The Five Sessions (audio quality varies):")
    for idx,s in enumerate(sessions, start=1):
        tone = s["tone"]
        snippet = s["transcript"].strip().replace("\n"," ")
        if len(snippet) > 220: snippet = snippet[:217] + "..."
        # friendly tone label mapping (optional)
        tone_map = {
            "confident":"clear audio (confident)",
            "static_interference":"static interference / noisy",
            "shouting":"shouting / loud",
            "whispered":"whispered / low volume",
            "emotional":"emotional / crying"
        }
        tone_desc = tone_map.get(tone, tone)
        summary_lines.append(f" • Session {idx} ({tone_desc}): “{snippet}”")
    summary_lines.append("\nYour Truth Weaver's Analysis:")
    # write placeholder analysis header (the JSON script will produce formal JSON)
    summary_lines.append(" - (See JSON output for structured findings.)")

    summary_text = "\n".join(summary_lines)
    summary_path = os.path.join(OUTPUT_DIR, f"{subj}_summary.txt")
    with open(summary_path, "w", encoding="utf-8") as fh:
        fh.write(summary_text)
    print("Saved summary text:", summary_path)


[atlas_2025] Processing atlas_2025_1.mp3 ...
[atlas_2025] Processing atlas_2025_2.mp3 ...
[atlas_2025] Processing atlas_2025_3.mp3 ...
[atlas_2025] Processing atlas_2025_4.mp3 ...
[atlas_2025] Processing atlas_2025_5.mp3 ...
Saved sessions JSON: /content/drive/MyDrive/Hackthon/Output/atlas_2025_sessions.json
Saved summary text: /content/drive/MyDrive/Hackthon/Output/atlas_2025_summary.txt
[eos_2023] Processing eos_2023_1.mp3 ...
[eos_2023] Processing eos_2023_2.mp3 ...
[eos_2023] Processing eos_2023_3.mp3 ...
[eos_2023] Processing eos_2023_4.mp3 ...
[eos_2023] Processing eos_2023_5.mp3 ...
Saved sessions JSON: /content/drive/MyDrive/Hackthon/Output/eos_2023_sessions.json
Saved summary text: /content/drive/MyDrive/Hackthon/Output/eos_2023_summary.txt
[hyperion_2022] Processing hyperion_2022_1.mp3 ...
[hyperion_2022] Processing hyperion_2022_2.mp3 ...
[hyperion_2022] Processing hyperion_2022_3.mp3 ...
[hyperion_2022] Processing hyperion_2022_4.mp3 ...
[hyperion_2022] Processing hyperion_

In [ ]:
"""
truth_weaver_json.py
- Reads <subject>_sessions.json (from Script 2) and produces final Truth Weaver JSON.
- Schema enforced exactly:
  {
    "shadow_id": "string",
    "revealed_truth": {
      "programming_experience": "string",
      "programming_language": "string",
      "skill_mastery": "string",
      "leadership_claims": "string",
      "team_experience": "string",
      "skills and other keywords": [list of strings]
    },
    "deception_patterns": [
      {
        "lie_type": "string",
        "contradictory_claims": [list of strings]
      }
    ]
  }
"""

import os, glob, json, re
from pathlib import Path

OUTPUT_DIR = "/content/drive/MyDrive/Hackthon/Output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LANGUAGES = ["python","java","c++","c#","javascript","typescript","go","rust","ruby","php",
             "swift","kotlin","scala","sql","rails","django","react","kubernetes"]

SKILLS = ["machine learning","deep learning","ai","artificial intelligence","cloud",
          "aws","gcp","azure","nlp","computer vision","devops","docker","postgres",
          "mysql","redis","cosmos db","networking","security"]

WORDS_TO_NUM = {
    "zero":0,"one":1,"two":2,"three":3,"four":4,"five":5,"six":6,
    "seven":7,"eight":8,"nine":9,"ten":10,"eleven":11,"twelve":12
}

def normalize_experience(claims):
    """Convert claims like 'six years', '3 years' into a normalized string."""
    months_list = []
    for c in claims:
        c = c.lower()
        # spelled number
        m_word = re.match(r"(" + "|".join(WORDS_TO_NUM.keys()) + r")\s+(years?|months?)", c)
        if m_word:
            num = WORDS_TO_NUM[m_word.group(1)]
            unit = m_word.group(2)
            months_list.append(num*12 if "year" in unit else num)
            continue
        # numeric
        m_num = re.match(r"(\d+)\s*(years?|months?)", c)
        if m_num:
            val = int(m_num.group(1))
            unit = m_num.group(2)
            months_list.append(val*12 if "year" in unit else val)

    if not months_list:
        return "unknown"
    mn, mx = min(months_list), max(months_list)
    def months_to_str(m):
        return f"{m//12} years" if m >= 12 and m % 12 == 0 else (f"{m} months" if m < 12 else f"{m/12:.1f} years")

    if mn == mx:
        return months_to_str(mn)
    return f"{months_to_str(mn)} - {months_to_str(mx)}"

def detect_languages(text):
    t = text.lower()
    return [lang for lang in LANGUAGES if re.search(r"\b" + re.escape(lang) + r"\b", t)]

def detect_skills(text):
    t = text.lower()
    return [sk.title() for sk in SKILLS if sk in t]

def detect_mastery(text):
    t = text.lower()
    if re.search(r"\b(expert|advanced|mastered)\b", t): return "advanced"
    if re.search(r"\b(intermediate|decent)\b", t): return "intermediate"
    if re.search(r"\b(beginner|learning|internship|just started|workshop)\b", t): return "beginner"
    return "unknown"

def detect_team_claims(text):
    t = text.lower()
    if re.search(r"\bled\b|\bmanager\b|\barchitect\b", t): return "claimed"
    if re.search(r"\bintern\b|\binternship\b|\bwork alone\b|\bjust watched\b", t): return "individual contributor"
    return "unknown"

# === MAIN ===
session_files = glob.glob(os.path.join(OUTPUT_DIR, "*_sessions.json"))
if not session_files:
    print("No session files found in", OUTPUT_DIR)

for sf in session_files:
    data = json.load(open(sf, "r", encoding="utf-8"))
    subj = data.get("shadow_id") or Path(sf).stem.replace("_sessions","")
    sessions = data.get("sessions", [])

    transcripts_all = " ".join([s.get("transcript","") for s in sessions])

    # Collect claims
    exp_claims = re.findall(r"(\d+\s*(?:years?|months?)|" + "|".join(WORDS_TO_NUM.keys()) + r"\s*(?:years?|months?))", transcripts_all.lower())
    languages = detect_languages(transcripts_all)
    skills = detect_skills(transcripts_all)
    mastery = detect_mastery(transcripts_all)
    team_claim = detect_team_claims(transcripts_all)

    # Build revealed_truth
    revealed = {
        "programming_experience": normalize_experience(exp_claims) if exp_claims else "unknown",
        "programming_language": languages[0] if languages else "unknown",
        "skill_mastery": mastery,
        "leadership_claims": "fabricated" if team_claim=="claimed" and "intern" in transcripts_all.lower() else team_claim,
        "team_experience": "individual contributor" if "intern" in transcripts_all.lower() or "work alone" in transcripts_all.lower() else team_claim,
        "skills and other keywords": list(set(skills))
    }

    # Detect contradictions
    deceptions = []
    if len(set(exp_claims)) > 1:
        deceptions.append({
            "lie_type": "experience_inflation",
            "contradictory_claims": list(set(exp_claims))
        })
    if "i'm not" in transcripts_all.lower() or "fraud" in transcripts_all.lower():
        deceptions.append({
            "lie_type": "emotional_breakdown",
            "contradictory_claims": ["self-doubt statements detected"]
        })

    # Final JSON
    result = {
        "shadow_id": subj,
        "revealed_truth": revealed,
        "deception_patterns": deceptions
    }

    outpath = os.path.join(OUTPUT_DIR, f"{subj}_result.json")
    with open(outpath, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)
    print("Saved:", outpath)


Saved: /content/drive/MyDrive/Hackthon/Output/eos_2023_result.json
Saved: /content/drive/MyDrive/Hackthon/Output/hyperion_2022_result.json
Saved: /content/drive/MyDrive/Hackthon/Output/oceanus_2022_result.json
Saved: /content/drive/MyDrive/Hackthon/Output/rhea_2024_result.json
Saved: /content/drive/MyDrive/Hackthon/Output/selene_2024_result.json


In [ ]:
"""
prepare_submission.py
- Prepares submission artifacts in /Output/Final:
  1. Transcript TXT files (all sessions per subject).
  2. Final JSON files (schema-compliant).
  3. Source Code ZIP archive.
"""

import os, glob, json, shutil, zipfile
from pathlib import Path

OUTPUT_DIR = "/content/drive/MyDrive/Hackthon/Output"
FINAL_DIR = os.path.join(OUTPUT_DIR, "Final")
os.makedirs(FINAL_DIR, exist_ok=True)

# === 1. Transcript Files (.txt) ===
session_files = glob.glob(os.path.join(OUTPUT_DIR, "*_sessions.json"))
for sf in session_files:
    data = json.load(open(sf, "r", encoding="utf-8"))
    subj = data.get("shadow_id") or Path(sf).stem.replace("_sessions","")
    sessions = data.get("sessions", [])

    transcript_path = os.path.join(FINAL_DIR, f"{subj}.txt")
    with open(transcript_path, "w", encoding="utf-8") as fh:
        for idx, s in enumerate(sessions, start=1):
            fh.write(f"=== SESSION {idx} | {subj} ===\n")
            fh.write(s.get("transcript","").strip() + "\n\n")
    print("Transcript saved:", transcript_path)

# === 2. Final JSON Files (.json) ===
result_files = glob.glob(os.path.join(OUTPUT_DIR, "*_result.json"))
for rf in result_files:
    dest = os.path.join(FINAL_DIR, Path(rf).name)
    shutil.copy(rf, dest)
    print("JSON saved:", dest)

# === 3. Source Code Archive (.zip) ===
zip_path = os.path.join(FINAL_DIR, "source_code.zip")
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for fname in ["train_tone_finetune.py","transcribe_and_label.py","truth_weaver_json.py"]:
        fpath = os.path.join("/content", fname) if os.path.exists(os.path.join("/content", fname)) else fname
        if os.path.exists(fpath):
            zipf.write(fpath, arcname=fname)
print("Source code archived:", zip_path)

print("\n✅ Submission prepared in:", FINAL_DIR)


Transcript saved: /content/drive/MyDrive/Hackthon/Output/Final/eos_2023.txt
Transcript saved: /content/drive/MyDrive/Hackthon/Output/Final/hyperion_2022.txt
Transcript saved: /content/drive/MyDrive/Hackthon/Output/Final/oceanus_2022.txt
Transcript saved: /content/drive/MyDrive/Hackthon/Output/Final/rhea_2024.txt
Transcript saved: /content/drive/MyDrive/Hackthon/Output/Final/selene_2024.txt
JSON saved: /content/drive/MyDrive/Hackthon/Output/Final/eos_2023_result.json
JSON saved: /content/drive/MyDrive/Hackthon/Output/Final/hyperion_2022_result.json
JSON saved: /content/drive/MyDrive/Hackthon/Output/Final/oceanus_2022_result.json
JSON saved: /content/drive/MyDrive/Hackthon/Output/Final/rhea_2024_result.json
JSON saved: /content/drive/MyDrive/Hackthon/Output/Final/selene_2024_result.json
Source code archived: /content/drive/MyDrive/Hackthon/Output/Final/source_code.zip

✅ Submission prepared in: /content/drive/MyDrive/Hackthon/Output/Final
